In [50]:
from tenpy.tools.process import mkl_set_nthreads, mkl_get_nthreads
import scipy.linalg as la
from tenpy.models.model import CouplingModel, NearestNeighborModel, MPOModel, CouplingMPOModel
from tenpy.networks.site import SpinHalfSite,BosonSite
import numpy as np
from tenpy.tools.params import asConfig
from tenpy.models.lattice import Site, Chain, MultiSpeciesLattice
from tenpy.linalg import np_conserved as npc
import tenpy.models.spins
import tenpy.networks.mps as mps
import tenpy.networks.site as site
from tenpy.algorithms import tdvp
from tenpy.networks.mps import MPS
import scipy.special
import copy
import time
import random
import matplotlib.pyplot as plt
import pandas as pd
from joblib import Parallel, delayed, dump, load
import time
import csv, ast
from os import listdir
from os.path import join
from tasks import *
tenpy.tools.optimization.set_level(3)
rng = np.random.default_rng(43)
np.random.seed(43)
random.seed(43)
class NewBosonSite(Site):
  def __init__(self, Nmax=1, delta_t=1, conserve='N', filling=0.):

    # Defining the conserve values
    if not conserve:
        conserve = 'None'
    if conserve not in ['N', 'parity', 'None']:
        raise ValueError("invalid `conserve`: " + repr(conserve))

    # Local Dimension of each Boson site
    dim = Nmax + 1
    states = [str(n) for n in range(0, dim)]
    if dim < 2:
        raise ValueError("local dimension should be larger than 1....")


    # ------------------------------------------------------------
        # Normalized time-bin annihilation operator b_k
        #
        # New convention:
        #     b_k = Delta A_k / sqrt(delta_t)
        #
        # Therefore:
        #     b_k |n> = sqrt(n) |n-1>
        #
        # and:
        #     [b_k, b_k^\dagger] = 1
    # ------------------------------------------------------------
    b = np.zeros([dim, dim], dtype=np.float64)  # destruction/annihilation operator
    for n in range(1, dim):
        b[n - 1, n] = np.sqrt(n)
    bd = np.transpose(b)  # .conj() wouldn't do anything
    dA = np.sqrt(delta_t) * b
    dAd = np.sqrt(delta_t) * bd
    # Note: np.dot(Bd, B) has numerical roundoff errors of eps~=4.4e-16.
    # ------------------------------------------------------------
        # Normalized quadratures:
        #
        #     Q_k = b_k + b_k^\dagger
        #     P_k = i(b_k^\dagger-b_k)
    # ------------------------------------------------------------
    theta = np.pi/3
    Q = (np.exp(-1j*theta)*b + np.exp(1j*theta)*bd) # first quadrature
    P = (np.exp(1j*theta)*bd - np.exp(-1j*theta)*b) * (1j) # second quadrature
    QQ = np.dot(Q, Q)
    PP = np.dot(P, P)

    #---- Vacuum Projection
    P0 = np.zeros([dim, dim], dtype=np.float64)
    P0[0, 0] = 1.0

    # Number operator
    Ndiag = np.arange(dim, dtype=np.float64)
    N = np.diag(Ndiag)
    NN = np.diag(Ndiag**2)
    dN = np.diag(Ndiag - filling)
    dNdN = np.diag((Ndiag - filling)**2)

      
    # Operator sets
    ops = dict(B=b, Bd=bd,dA=dA,dAd=dAd, Q=Q, P=P,P0=P0, PP=PP, QQ=QQ, N=N, NN=NN, dN=dN, dNdN=dNdN)
      
    if conserve == 'N':
        chinfo = npc.ChargeInfo([1], ['N'])
        leg = npc.LegCharge.from_qflat(chinfo, range(dim))
    elif conserve == 'parity':
        chinfo = npc.ChargeInfo([2], ['parity_N'])
        leg = npc.LegCharge.from_qflat(chinfo, [i % 2 for i in range(dim)])
    else:
        leg = npc.LegCharge.from_trivial(dim)
        
    self.Nmax = Nmax
    self.conserve = conserve
    self.filling = filling
    Site.__init__(self, leg, states, sort_charge=True, **ops)
    self.state_labels['vac'] = self.state_labels['0']  # alias
    self.charge_to_JW_parity = np.array([0] * leg.chinfo.qnumber, int)  # trivial

  def __repr__(self):
    """Debug representation of self."""
    return "BosonSite({N:d}, {c!r}, {f:f})".format(N=self.Nmax,
                                                    c=self.conserve,
                                                    f=self.filling)


In [51]:

class AtomMirror(CouplingModel, MPOModel):
  def __init__(self, model_params, Driver):
    # 0) read out/set default parameters
    model_params = asConfig(model_params, "AtomMirror")
    delta_t = model_params.get('delta_t', 1)
    delay_steps = model_params.get('delay_steps', 1)
    Delta = model_params.get('Delta', 0) # Default: omega_L - omega = Delta, it is 0 for on resonant laser
    gamma = model_params.get('gamma', 1.) # Atom decaying rate
    Omega = model_params.get('Omega', 1.5) # Input
    phi = model_params.get('phi', 0.)
    max_photon = model_params.get('max_photon', 3)
    #Driver = model_params.get('input', 0)
    hbar = 1
    self.bc = 'finite'
    # 1-3):
    # 1) charges of the physical leg. The only time that we actually define charges!
    leg = tenpy.linalg.np_conserved.LegCharge.from_trivial(2) # non defined charge
    # 2) onsite operators
    USE_PREDEFINED_SITE = False
    if not USE_PREDEFINED_SITE:
      Sp = [[0., 1.], [0., 0.]]
      Sm = [[0., 0.], [1., 0.]]
      Sx = np.array(Sp) + np.array(Sm)
      Sy = 1j*(np.array(Sp)-np.array(Sm))
      Sxx = np.dot(Sx,Sx)
      Syy = np.dot(Sy,Sy)
      Sz = [[1, 0.], [0., -1]]
      Id = [[1,0],[0,1]]

    # (Can't define Sx and Sy as onsite operators: they are incompatible with Sz charges.)
    # 3) local physical site
      system_site = Site(leg, ['up', 'down'], Sp=Sp, Sm=Sm, Sz=Sz, Sx=Sx, Sy=Sy, Sxx=Sxx,Syy=Syy)
    else:
      system_site = SpinHalfSite(conserve=None)
    # Define time-bin sites (each bin can hold N photon state)
    if not USE_PREDEFINED_SITE:
      time_bin_site = NewBosonSite(Nmax=max_photon,delta_t=delta_t,conserve=None)
    else:
      time_bin_site = BosonSite(Nmax=max_photon,conserve=None)  # Modify as needed for bosonic modes

    sites = [time_bin_site]*2 + [system_site] + [time_bin_site for _ in range(delay_steps)] # counter . L-3
    tenpy.networks.site.set_common_charges([system_site, time_bin_site], new_charges='drop') # drop the charges
    # 4) lattice
    # Construct the initial MPS (vacuum state for time bins, initial state for the system)
    lattice = Chain(1, time_bin_site, bc="periodic", bc_MPS="finite") # Create the base lattice
    lat = MultiSpeciesLattice(lattice,sites)  # Create the actual lattice
    # 5) initialize CouplingModel
    CouplingModel.__init__(self, lat)
    # 6) add terms of the Hamiltonian
    # adding the atom hamiltonian
    self.add_onsite(-hbar*Delta/2*delta_t, 2, 'Sz')
    self.add_onsite(-hbar*Delta/2*delta_t, 2, 'Id')
    self.add_onsite(-hbar/2*Omega*delta_t*Driver, 2, 'Sm') 
    self.add_onsite(-hbar/2*Omega*delta_t*Driver, 2, 'Sp')
    #adding the coupling term
    self.add_coupling(-1j*hbar*np.sqrt(gamma/2)*np.exp(-1j*phi), 2-1, 'dA', 2, 'Sp',1,plus_hc=True) # add coupling between atom site and current k site
    self.add_coupling(-1j*hbar*np.sqrt(gamma/2), 2, 'Sp', 2+1, 'dA',1, plus_hc=True) # add coupling between atom site and past site k-delay
    # the `plus_hc=True` adds the h.c. term
    # 7) initialize H_MPO
    MPOModel.__init__(self, lat, self.calc_H_MPO()) # initialize MPO for this hamiltonian

In [52]:
class System:
    def __init__(self, model_params, engine_params, train_params):
        ### obtain model parameters
        self.model_params = asConfig(model_params, "AtomMirror")
        self.engine_params = asConfig(engine_params, "Engine")
        #numerical step
        self.delta_t = model_params.get('delta_t')
        # real delay time to time bins
        self.delay_steps = model_params.get('delay_steps')
        self.phi = model_params.get('phi')
        self.delay_steps1 = self.delay_steps
        self.swap_steps = self.delay_steps-1
        # max photon
        self.photon_number = model_params.get('max_photon')
        # max bin
        self.max_bin = model_params.get('max_bin')
        # Input counter
        self.time_counter = 0
        # total data points
        self.t_max = model_params.get('t_max')
        self.data_points = int(self.t_max/self.delta_t)
        self.t_sample = model_params.get('t_sample')
        # runtime option
        self.option = engine_params.get('run_option')
        print(model_params)
        ### Train Params
        self.train_points = train_params.get('train_points')
        self.delay_points = train_params.get('delay_points')
        self.observation = train_params.get('observation')
        self.nodes = train_params.get('nodes')
        self.prediction_delay = train_params.get('prediction_delay')
        self.task_name = train_params.get('task')
        self.fold_num = train_params.get('fold_num')
        self.type = train_params.get('type')
        print(self.model_params)
        # generating the task

        
        self.task, y_bar = sine_square_input_task(Nsegments = 110, wsin= 10.0, Nsin = self.t_sample, seed = 1234)
        self.task = self.repeat_mask(self.task,2)
        # initilize the state
        self.psi = self.initial_state()
        # holders for calculations
        self.num_reset = 0
        ### for atoms
        self.Es = []
        self.N_total = []
        self.Sx = []
        self.Sy = []
        self.Sxx = []
        self.Syy = []
        ### for time bins
        fixed_bin_index = [i for i in range(-(self.delay_steps),1)] # storing all values from current time down to coming back field
        PQ_keys = ["PQ" + str(i) for i in fixed_bin_index]
        self.operators = {key: [] for key in (PQ_keys)}
        
        self.filename = f'SS{self.t_sample}sampled_points_rep1' +  '.csv'
        self.write_header = True
        
    def repeat_mask(self, values, N):
        return np.repeat(values, N)
        
    def initial_state(self):
        """Done"""
        leg = tenpy.linalg.np_conserved.LegCharge.from_trivial(2)
        Sp = [[0., 1.], [0., 0.]]
        Sm = [[0., 0.], [1., 0.]]
        Sx = np.array(Sp) + np.array(Sm)
        Sy = 1j*(np.array(Sp)-np.array(Sm))
        Sxx = np.dot(Sx,Sx)
        Syy = np.dot(Sy,Sy)
        Sz = [[1, 0.], [0., -1]]
        Id = [[1, 0.], [0., 1]]
        system_site = Site(leg, ['up', 'down'], Sp=Sp, Sm=Sm, Sz=Sz, Sx=Sx, Sy=Sy, Sxx=Sxx,Syy=Syy) # theoretically sx and sy are not compatible with sz
        time_bin_site = NewBosonSite(Nmax=self.photon_number,delta_t=self.delta_t,conserve=None)  # Modify as needed for bosonic modes
        sites = [time_bin_site]*2 + [system_site] + [time_bin_site for _ in range(self.delay_steps)]
        initial_state = ['vac']*2 + ['down'] + ['vac'] * (self.delay_steps)
        # Create the matrix product state
        psi = MPS.from_product_state(sites, initial_state, "finite")
        print("Initial state is initiated")
        return psi

    def execution(self, option):
        if option == 'speed' and self.delay_steps > self.max_bin:
            # 1. Swap to bring delayed bins near the system
            for i in reversed(range(self.swap_steps)): #a
                self.psi.swap_sites(2+i+1)
            # 2. Apply the interaction unitary
            
            self.tdvp_engine.run()
            
            Lshape = self.psi._B[-2].shape[0]
            Mshape = self.psi._B[-2].shape[1]
            Rshape = self.psi._B[-1].shape[2]
            data = np.random.rand(Lshape, Mshape, Rshape) * 0
            data[0,0,0] = 1 # ground state is assumed as 1.
            
            arr = npc.Array.from_ndarray_trivial(data, labels=['vL', 'p', 'vR'])
            Bs = self.psi._B[0:1] + self.psi._B[0:1] + self.psi._B[1:-2] + [arr]
            sites = self.psi.sites[0:1] + self.psi.sites[0:1] + self.psi.sites[1:-1] 
            Svs = self.psi._S[0:1] + self.psi._S[0:1]+ self.psi._S[1:-2] + self.psi._S[0:1]
            self.psi = MPS(sites, Bs, Svs, bc='finite', form='B', norm=1.0)
            
            for i in range(self.swap_steps):
                self.psi.swap_sites(2+i+2)

            print("len is", len(self.psi.chi))
            self.measure()
            self.psi.swap_sites(2)
        else:
            """accuracy is traded of with runtime"""
            # 1. Swap to bring delayed bins near the system
            for i in reversed(range(self.swap_steps)): #a
                self.psi.swap_sites(2+i+1)
            # 2. Apply the interaction unitary
            self.tdvp_engine.run()
            self.psi = MPS(self.psi.sites[0:1] + self.psi.sites[0:1] + self.psi.sites[1:],
                    self.psi._B[0:1] + self.psi._B[0:1] + self.psi._B[1:],
                    self.psi._S[0:1] + self.psi._S[0:1]+ self.psi._S[1:], bc='finite', form='B', norm=1.0)
            for i in range(self.swap_steps):
                self.psi.swap_sites(2+i+2)

            print("len is", len(self.psi.chi))
            
            self.measure()
            self.psi.swap_sites(2)
            # update the steps on the right of atoms
            self.delay_steps +=1
            self.model_params['delay_steps'] = self.delay_steps
    

    def measure(self):
        rho_i = self.psi.get_rho_segment([2+1]) # the system is located at 3    
        up_idx = self.psi.sites[2+1].state_labels['up'] # measure the excitation
        self.Es.append(rho_i[up_idx, up_idx]) # Population

        ### atom's quadratures
        self.Sx.append(self.psi.expectation_value('Sx',[2+1]))
        self.Sxx.append(self.psi.expectation_value('Sxx',[2+1]))
        self.Sy.append(self.psi.expectation_value('Sy',[2+1]))
        self.Syy.append(self.psi.expectation_value('Syy',[2+1]))

        ### Operator's quadratures
        for key, value_list in self.operators.items():
            if 0 == int(key[2:]): # For index 0, we measure the field on the left of atom which is at 2
                v = (self.psi.expectation_value('P',[2]),self.psi.expectation_value('Q',[2]))
                value_list.append(v)
            elif self.option == 'speed' and self.delay_steps >= self.max_bin:  # For other index, -1,-2,-3, we measure the sites for how many steps from the atom at index 3
                v = (self.psi.expectation_value('P',[3-int(key[2:])]), self.psi.expectation_value('Q',[3-int(key[2:])]))
                value_list.append(v)
            else: # Start applying special treatment for the speed
                v = (self.psi.expectation_value('P',[3-int(key[2:])]), self.psi.expectation_value('Q',[3-int(key[2:])]))
                value_list.append(v)

        self.save_large_csv(self.filename, self.Es, self.Sx, self.Sxx, self.Sy, self.Syy, self.operators)
        
    def run(self):
        Driver = self.task[self.time_counter]
        model = AtomMirror(self.model_params, Driver)
        self.tdvp_engine = tdvp.TwoSiteTDVPEngine(self.psi, model, self.engine_params)
        print(self.delay_steps)
        self.execution(self.option)
        print(self.model_params)
        for i in range(self.data_points-1):
            self.time_counter +=1
            Driver = self.task[self.time_counter]
            atom_mirror = AtomMirror(self.model_params, Driver)
            self.tdvp_engine = tdvp.TwoSiteTDVPEngine(self.psi, atom_mirror, self.engine_params)          
            self.execution(self.option)

    def row_generator(self,Es, Sx, Sxx, Sy, Syy, operators):
        i = self.time_counter
        row = {
            'Es': Es[i],
            'Sx': Sx[i],
            'Sxx': Sxx[i],
            'Sy': Sy[i],
            'Syy': Syy[i],
        }
        for key, values in operators.items():
            row[key] = values[i]
        return row
        
    def save_large_csv(self, filename, Es, Sx, Sxx, Sy, Syy, operators):
        keys = ['Es', 'Sx', 'Sxx', 'Sy', 'Syy'] + list(operators.keys())
        
        with open(filename, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=keys) 
            if self.write_header:
                writer.writeheader()
                self.write_header = False
                pass
            
            row = self.row_generator(Es, Sx, Sxx, Sy, Syy, operators)
            writer.writerow(row)
                
            
    def fold_run(self):
        self.run()


In [53]:
def t_sample_points(t_samples):
    model_params_list = []
    for t_sample in t_samples:
        model_params['t_sample'] = t_sample
        model_params['t_max'] = model_params['t_max']*t_sample
        model_params_list.append(model_params.copy())
    return model_params_list

def fold_task(model_params):
    obj = System(model_params, tdvp_params, train_params)  # fresh instance
    return obj.fold_run()


In [54]:
# scaled with gamma
tau = 10
t_sample = 8
t_rep = 1
t_max = 110*t_rep
Delta = 0
gamma = .1
Omega = .15
phi = np.pi/3
max_photon = 2
max_bin = 30
delta_t = 1
delay_steps = int(tau/1)
chi_max = 5
train_points = 1
delay_points = 1
prediction_delay = 1
observation = train_points + delay_points
nodes = int(tau/delta_t)
task1 = 'ASR'
fold_num = 2
input_type = 'mean'
model_params = {
      'tau': tau,
      't_max': t_max,  
      't_sample':t_sample,
      'Delta': Delta,
      'gamma': gamma,
      'Omega' : Omega,
      'phi': phi,
      'max_photon': max_photon,
      'max_bin' : max_bin,
      'delta_t': delta_t, 
      'delay_steps': delay_steps, 
  }

tdvp_params = {
    'start_time': 0,
    'run_option': 'speed',
    'dt': 1,
    'trunc_params': {
        'chi_max': chi_max,
        'svd_min': 1.e-10,
        'trunc_cut': None
    },
    'N_steps': 1,
    "max_N_sites_per_ring" : 10000
  }

train_params = {
    'train_points' : train_points,
    'delay_points' : delay_points,
    'observation' : observation, # we observe results from [observation:]
    'nodes' : nodes,
    'prediction_delay': prediction_delay,
    'task' : task1,
    'fold_num' : fold_num,
    'type': input_type,
}

t_samples= [8]#,8,12,16,20]
train_params_list = t_sample_points(t_samples)
results = Parallel(n_jobs=1)(delayed(fold_task)(model_params) for model_params in train_params_list)


{'tau': 10, 't_max': 880, 't_sample': 8, 'Delta': 0, 'gamma': 0.1, 'Omega': 0.15, 'phi': 1.0471975511965976, 'max_photon': 2, 'max_bin': 30, 'delta_t': 1, 'delay_steps': 10}
Config, name='AtomMirror', options:
{'Delta': 0,
 'Omega': 0.15,
 'delay_steps': 10,
 'delta_t': 1,
 'gamma': 0.1,
 'max_bin': 30,
 'max_photon': 2,
 'phi': 1.0471975511965976,
 't_max': 880,
 't_sample': 8,
 'tau': 10}
Initial state is initiated
10
len is 13
Config, name='AtomMirror', options:
{'Delta': 0,
 'Omega': 0.15,
 'delay_steps': 11,
 'delta_t': 1,
 'gamma': 0.1,
 'max_bin': 30,
 'max_photon': 2,
 'phi': 1.0471975511965976,
 't_max': 880,
 't_sample': 8,
 'tau': 10}
len is 14
len is 15
len is 16
len is 17
len is 18
len is 19
len is 20
len is 21
len is 22
len is 23
len is 24
len is 25
len is 26
len is 27
len is 28
len is 29
len is 30
len is 31
len is 32
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is 33
len is

C:\Users\Phi Hung Nguyen\anaconda3\Lib\site-packages\tenpy\tools\params.py:232: UserWarning: unused options for config AtomMirror:
['max_bin', 't_max', 't_sample', 'tau']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
C:\Users\Phi Hung Nguyen\anaconda3\Lib\site-packages\tenpy\tools\params.py:232: UserWarning: unused option ['run_option'] for config Engine
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))
